<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/Industrial_Model_Mini_Mill_Optimized_TwoDay_Real48h.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================
# MINI-MILL TWO-DAY HORIZON OPTIMIZATION
# REAL 48-HOUR PRICE INPUT VERSION
# Input file:
#   Production sequense to Prices 19March 2025 set.xlsx
# =========================================================

input_file = "Production sequense to Prices 19March 2025 set.xlsx"
output_file = "Industrial_Model_Mini_Mill_Optimized_TwoDay_Real48h.xlsx"

# -----------------------------
# 1. READ INPUT
# -----------------------------
raw = pd.read_excel(input_file)

if raw.empty:
    raise ValueError("Input file is empty.")

raw.columns = [str(c).strip() for c in raw.columns]

# -----------------------------
# 2. FLEXIBLE PARSING
# -----------------------------
if {"Day", "Hour", "Price"}.issubset(set(raw.columns)):
    df_prices = raw[["Day", "Hour", "Price"]].copy()
else:
    # fallback: first 2 columns = Hour, Price
    if raw.shape[1] < 2:
        raise ValueError("Input file must contain at least 2 columns.")

    df_prices = raw.iloc[:, :2].copy()
    df_prices.columns = ["Hour", "Price"]

    if len(df_prices) != 48:
        raise ValueError(
            "For this real 48-hour version, the file must contain exactly 48 rows "
            "(24 for Day 1 + 24 for Day 2), unless it has explicit Day, Hour, Price columns."
        )

    df_prices["Day"] = ["Day 1"] * 24 + ["Day 2"] * 24
    df_prices = df_prices[["Day", "Hour", "Price"]]

df_prices = df_prices.dropna(subset=["Day", "Hour", "Price"]).copy()
df_prices["Hour"] = pd.to_numeric(df_prices["Hour"], errors="raise").astype(int)
df_prices["Price"] = pd.to_numeric(df_prices["Price"], errors="raise").astype(float)
df_prices["Day"] = df_prices["Day"].astype(str).str.strip()

def normalize_day_label(x):
    x_clean = str(x).strip().lower()
    if x_clean in {"1", "day1", "day 1", "d1"}:
        return "Day 1"
    if x_clean in {"2", "day2", "day 2", "d2"}:
        return "Day 2"
    return str(x)

df_prices["Day"] = df_prices["Day"].apply(normalize_day_label)

valid_days = {"Day 1", "Day 2"}
if not set(df_prices["Day"]).issubset(valid_days):
    raise ValueError("Day column must contain only Day 1 and Day 2, or equivalent labels.")

counts = df_prices["Day"].value_counts().to_dict()
if counts.get("Day 1", 0) != 24 or counts.get("Day 2", 0) != 24:
    raise ValueError("Input must contain exactly 24 rows for Day 1 and 24 rows for Day 2.")

df_day1 = df_prices[df_prices["Day"] == "Day 1"].sort_values("Hour").reset_index(drop=True)
df_day2 = df_prices[df_prices["Day"] == "Day 2"].sort_values("Hour").reset_index(drop=True)

expected_hours = list(range(24))
if df_day1["Hour"].tolist() != expected_hours:
    raise ValueError("Day 1 must contain hours 0 to 23 exactly once.")
if df_day2["Hour"].tolist() != expected_hours:
    raise ValueError("Day 2 must contain hours 0 to 23 exactly once.")

df_two_day = pd.concat([df_day1, df_day2], ignore_index=True)
df_two_day.insert(0, "Chronological Hour", range(48))

# -----------------------------
# 3. FIXED 24-STAGE SEQUENCE
# -----------------------------
stage_sequence = [
    ("Step 1", "EAF1.1", 72, 1),
    ("Step 1", "EAF1.2", 78, 2),
    ("Step 1", "EAF1.3", 80, 3),
    ("Step 1", "EAF1.4", 72, 4),

    ("Step 2", "Secondary Downstream 1.1", 32, 5),
    ("Step 2", "Secondary Downstream 1.2", 26, 6),
    ("Step 2", "Secondary Downstream 1.3", 20, 7),

    ("Step 3", "SD sequence Blue Colour", 15, 8),

    ("Step 4", "Minimum Critical Load 1", 12, 9),
    ("Step 4", "Minimum Critical Load 2", 12, 10),
    ("Step 4", "Minimum Critical Load 3", 12, 11),
    ("Step 4", "Minimum Critical Load 4", 12, 12),
    ("Step 4", "Minimum Critical Load 5", 12, 13),
    ("Step 4", "Minimum Critical Load 6", 12, 14),

    ("Step 5", "Preparation Blue 1", 15, 15),
    ("Step 5", "Preparation Blue 2", 24, 16),
    ("Step 5", "Preparation Blue 3", 38, 17),

    ("Step 6", "EAF2.1", 72, 18),
    ("Step 6", "EAF2.2", 78, 19),
    ("Step 6", "EAF2.3", 80, 20),
    ("Step 6", "EAF2.4", 76, 21),

    ("Step 7", "Secondary Downstream 2.1", 28, 22),
    ("Step 7", "Secondary Downstream 2.2", 24, 23),
    ("Step 7", "Secondary Downstream 2.3", 20, 24),
]

df_sequence = pd.DataFrame(
    stage_sequence,
    columns=["Step", "Production Phase", "Total Load (MWh)", "t"]
)

# -----------------------------
# 4. BUILD SCHEDULE
# -----------------------------
def build_schedule_for_start(start_hour_day1, df_48, sequence):
    rows = []
    total_cost = 0.0

    for i, (step, phase, load, t) in enumerate(sequence):
        chrono_hour = start_hour_day1 + i
        rec = df_48.iloc[chrono_hour]

        price = float(rec["Price"])
        day = rec["Day"]
        hour_of_day = int(rec["Hour"])
        cost = load * price
        total_cost += cost

        rows.append({
            "t": t,
            "Step": step,
            "Production Phase": phase,
            "Total Load (MWh)": load,
            "Chronological Hour": chrono_hour,
            "Day": day,
            "Hour of Day": hour_of_day,
            "Price": price,
            "Cost = Load x Price": cost
        })

    return pd.DataFrame(rows), total_cost

# -----------------------------
# 5. OPTIMIZATION
# -----------------------------
optimization_rows = []
best_start = None
best_cost = None
best_schedule = None

for h in range(24):
    schedule_df, total_cost = build_schedule_for_start(h, df_two_day, stage_sequence)

    end_chrono = h + 23
    end_row = df_two_day.iloc[end_chrono]

    optimization_rows.append({
        "Candidate Start Hour in Day 1 (EAF1.1)": h,
        "Start Day": "Day 1",
        "End Chronological Hour": end_chrono,
        "End Day": end_row["Day"],
        "End Hour of Day": int(end_row["Hour"]),
        "Total Cost": total_cost
    })

    if best_cost is None or total_cost < best_cost:
        best_cost = total_cost
        best_start = h
        best_schedule = schedule_df.copy()

df_optimization = pd.DataFrame(optimization_rows)

# -----------------------------
# 6. SUMMARY + NOTES
# -----------------------------
best_end_chrono = best_start + 23
best_end_row = df_two_day.iloc[best_end_chrono]

df_summary = pd.DataFrame({
    "Metric": [
        "Optimal Step 1 start hour in Day 1 (EAF1.1)",
        "Start Day",
        "Sequence End Chronological Hour",
        "Sequence End Day",
        "Sequence End Hour of Day",
        "Minimum Total Cost",
        "Optimization Criterion",
        "Number of Candidate Start Hours"
    ],
    "Value": [
        best_start,
        "Day 1",
        best_end_chrono,
        best_end_row["Day"],
        int(best_end_row["Hour"]),
        best_cost,
        "Minimize sum of (Total Load × real electricity price) over 24 consecutive process-hours",
        24
    ]
})

df_notes = pd.DataFrame({
    "Notes": [
        "This version reads real 48-hour prices from Production sequense to Prices 19March 2025 set.xlsx.",
        "The optimization chooses the start hour of EAF1.1 only within Day 1.",
        "Once the start hour is selected, the remaining process stages follow chronologically.",
        "The t-index is process-relative: t=1 for EAF1.1 and t=24 for Secondary Downstream 2.3.",
        "Day 2 prices are taken directly from the input file."
    ]
})

# -----------------------------
# 7. WRITE WORKBOOK
# -----------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_day1.to_excel(writer, sheet_name="Input Prices Day1", index=False)
    df_day2.to_excel(writer, sheet_name="Input Prices Day2", index=False)
    df_two_day.to_excel(writer, sheet_name="Two-Day Prices", index=False)
    df_sequence.to_excel(writer, sheet_name="Stage Sequence", index=False)
    df_optimization.to_excel(writer, sheet_name="Optimization", index=False)
    best_schedule.to_excel(writer, sheet_name="Optimized Schedule", index=False)
    df_notes.to_excel(writer, sheet_name="Notes", index=False)

# -----------------------------
# 8. FORMAT WORKBOOK
# -----------------------------
wb = load_workbook(output_file)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

phase_fills = {
    "EAF": PatternFill("solid", fgColor="FCE4D6"),
    "Secondary": PatternFill("solid", fgColor="E2F0D9"),
    "SD sequence Blue Colour": PatternFill("solid", fgColor="D9EAF7"),
    "Minimum Critical Load": PatternFill("solid", fgColor="F4CCCC"),
    "Preparation Blue": PatternFill("solid", fgColor="D9E1F2"),
}

for ws in wb.worksheets:
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="center")

    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 40)

for sheet_name in [
    "Input Prices Day1", "Input Prices Day2", "Two-Day Prices",
    "Optimization", "Optimized Schedule", "Summary"
]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and ("Price" in str(header) or "Cost" in str(header)):
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_idx).number_format = "0.00"

ws_opt = wb["Optimized Schedule"]
headers_opt = [c.value for c in ws_opt[1]]
phase_col = headers_opt.index("Production Phase") + 1

for row in range(2, ws_opt.max_row + 1):
    phase_value = str(ws_opt.cell(row=row, column=phase_col).value)

    fill_to_use = None
    if phase_value.startswith("EAF"):
        fill_to_use = phase_fills["EAF"]
    elif phase_value.startswith("Secondary"):
        fill_to_use = phase_fills["Secondary"]
    elif phase_value == "SD sequence Blue Colour":
        fill_to_use = phase_fills["SD sequence Blue Colour"]
    elif phase_value.startswith("Minimum Critical Load"):
        fill_to_use = phase_fills["Minimum Critical Load"]
    elif phase_value.startswith("Preparation Blue"):
        fill_to_use = phase_fills["Preparation Blue"]

    if fill_to_use:
        ws_opt.cell(row=row, column=phase_col).fill = fill_to_use

ws_optim = wb["Optimization"]
headers_optim = [c.value for c in ws_optim[1]]
start_col = headers_optim.index("Candidate Start Hour in Day 1 (EAF1.1)") + 1

for row in range(2, ws_optim.max_row + 1):
    if ws_optim.cell(row=row, column=start_col).value == best_start:
        for col in range(1, ws_optim.max_column + 1):
            ws_optim.cell(row=row, column=col).fill = PatternFill("solid", fgColor="FFF2CC")

wb.save(output_file)

print(f"Output saved to: {output_file}")
print(f"Optimal Step 1 start hour (Day 1): {best_start}")
print(f"Optimal total cost: {best_cost:,.2f}")

Output saved to: Industrial_Model_Mini_Mill_Optimized_TwoDay_Real48h.xlsx
Optimal Step 1 start hour (Day 1): 8
Optimal total cost: 74,429.19
